# 📗 GraphRAG: 검색기·리랭킹·커뮤니티 필터·질의응답

지난 시간에는 논문을 그래프에 얹고, 임베딩으로 찾고, 개체와 이어 두었습니다. 이번 시간에는 그 위에서 **검색을 조립**합니다. 라이브러리 검색기로 묶고, 33·34일차의 **PageRank 와 커뮤니티**를 순위에 얹고, 찾은 근거로 **모델이 답을 만들게** 합니다. 그 답을 어디까지 믿을 수 있는지까지 봅니다.

## ⏪ 복습: 지난 시간까지

- **벡터 검색**: 논문 임베딩을 적재하고 `SEARCH` 절로 뜻이 가까운 논문을 찾았습니다.
- **전문 검색**: 글자가 든 논문을 정확히 찾았습니다(역할이 다릅니다).
- **`MENTIONS`**: 논문을 그 논문이 말한 개체 노드와 이어 두었습니다. **그래프로 넘어갈 다리**입니다.
- **GDS**(33·34일차): **PageRank**(중요한 노드)와 **커뮤니티**(촘촘히 이어진 묶음)를 구하는 법. 오늘 **2-1** 과 **3-1** 에서 이 그래프 위에 다시 계산해 검색에 씁니다.

**오늘의 목표**

오늘 배우는 것은 두 덩어리입니다.

| | 무엇인가 | 어디서 |
|---|---|---|
| **기본 GraphRAG** | 질문으로 문서를 찾고, 그 문서가 그래프에서 가리키는 개체를 함께 붙여 근거로 만들고, 모델이 그 근거 안에서 답한다 | 1절 · 4절 |
| **GraphRAG 검색 최적화** | 그 검색이 고른 후보를 그래프 구조로 **다시 세우거나**(리랭킹) **잘라 낸다**(커뮤니티 필터) | 2절 · 3절 |

4절에서 **최적화 없이 한 번 답한 뒤**, 최적화를 얹어 답이 어떻게 달라지는지 견줍니다.

**1. 검색기 조립** (기본 GraphRAG)
- [ ] (1-1) **`VectorRetriever`** 로 의미 검색을 조립하고, 결과의 모양을 `result_formatter` 로 정한다.
- [ ] (1-2) 같은 기호를 **벡터·전문 두 인덱스**에 나란히 걸어, 한쪽만 쓰면 무엇을 놓치는지 본다.

**2. PageRank 리랭킹** (검색 최적화)
- [ ] (2-1) **33일차**의 **투영**을 다시 만들고 **PageRank** 를 개체에 새긴다.
- [ ] (2-2) 개체 점수를 **문서 점수로 옮긴다**(그 문서가 언급한 개체들의 평균).
- [ ] (2-3) **눈금을 맞춰 두 점수를 섞고**, 섞는 비율이 순위를 어떻게 바꾸는지 잰다.

**3. 커뮤니티 필터** (검색 최적화)
- [ ] (3-1) **Leiden** 으로 개체를 묶고 **크기 분포**로 읽는다(묶음 수만 읽으면 오해한다).
- [ ] (3-2) 기준 약이 든 묶음에 무엇이 함께 있는지 들여다본다.
- [ ] (3-3) 그 묶음으로 **검색 범위를 좁히고**, 리랭킹과 하는 일이 어떻게 다른지 구분한다.

**4. 찾은 근거로 답 만들기** (기본 GraphRAG)
- [ ] (4-1) 논문 발췌에 **그 논문이 그래프에서 가리키는 개체**를 붙여 근거를 만든다.
- [ ] (4-2) 그 근거로 **모델이 답하게** 한다.
- [ ] (4-3) 답의 **인용을 대조**하고, 그 답이 무엇을 말할 수 있고 무엇을 말할 수 없는지 가른다.

> **데이터 출처**: 이 단원의 데이터는 **공개된 원본을 값 그대로** 쓴 것입니다.
>
> | 데이터 | 원본 | 이용 조건 |
> |---|---|---|
> | 논문 69편 (`pmc_docs.jsonl`) | PubMed Central Open Access Subset. 각 행의 `pmcid` 가 원문 주소다 | **CC BY** |
> | 의료 지식그래프 (`hetionet_*.csv`) | Hetionet v1.0 (https://het.io) 에서 CC0 출처만 골라낸 부분 | **CC0** |
> | 이름 사전 (`name2id.json.gz`) | 위 Hetionet 이름 + RxNav(미국 국립의학도서관) 약물 동의어 | CC0 · NLM |
>
> 지식그래프는 **2016년에 정리된 자료**이고, 논문은 최근 것입니다. 그래서 이 둘을 이어 붙이면 그래프가 모르는 사실이 논문 쪽에 있습니다. 이 단원은 그 상태 그대로 검색합니다.
>
> 그리고 **논문이 보고했다**와 **효능이 입증됐다**는 다릅니다. 검색으로 찾은 문장을 답으로 옮길 때 이 구분을 놓치면, 근거가 있는 것처럼 보이는 틀린 답이 나옵니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 임베딩 준비: embed_texts(문장 리스트) 가 768차원 OpenAI 임베딩을 돌려줍니다(실행만 하세요).
# embed_texts(문서 리스트) 는 계산한 벡터를 data/emb_cache.pkl 에 저장해 두고 다시 씁니다(69편을 매번 다시 부르지 않으려고요).
# embed_query(질문 한 문장) 는 저장하지 않습니다. 질문은 매번 새로 만드는 것이 표준입니다.
import pickle
from pathlib import Path

from langchain_openai import OpenAIEmbeddings

EMBED_MODEL = "text-embedding-3-large"
EMBED_DIM = 768                    # 3072차원으로 나오는 모델을 768차원으로 잘라 받는다(아래 dimensions)
_EMB_FILE = Path("data/emb_cache.pkl") if Path("data").exists() else Path("../data/emb_cache.pkl")
_EMB_CACHE = pickle.loads(_EMB_FILE.read_bytes()) if _EMB_FILE.exists() else {}   # {문서: 벡터}
embedder = OpenAIEmbeddings(model=EMBED_MODEL, dimensions=EMBED_DIM)


def embed_texts(texts):
    """문서 리스트 -> 768차원 임베딩 리스트. 저장된 것은 그대로 쓰고, 없는 것만 임베딩해 저장한다."""
    new = [t for t in texts if t not in _EMB_CACHE]                  # 저장돼 있지 않은 문서만 고른다
    if new:
        _EMB_CACHE.update(zip(new, embedder.embed_documents(new)))   # 실제 호출은 이 줄뿐
        _EMB_FILE.write_bytes(pickle.dumps(_EMB_CACHE))              # 통째로 다시 저장
    return [_EMB_CACHE[t] for t in texts]


def embed_query(text):
    """질문 한 문장 -> 768차원 임베딩. 질문은 저장하지 않는다(매번 새로 만든다)."""
    return embedder.embed_query(text)


print("임베딩 모델:", EMBED_MODEL, f"({EMBED_DIM}차원) / 저장된 문서:", len(_EMB_CACHE), "건")

In [ ]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 이 셀은 실행만 하세요.
# 몇 번이든 처음부터 다시 돌릴 수 있게 전부 내립니다. 반드시 "실습 전용" DB 여야 합니다.

# 1) GDS 투영: 노드를 지우기 전에 먼저 내린다. 투영은 원본 노드 id 를 기억하고 있어, 원본을 먼저 지우면 갈 곳을 잃는다
for _g in run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"):
    run_cypher("CALL gds.graph.drop($name) YIELD graphName RETURN graphName", name=_g["graphName"])

# 2) 노드와 관계
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH: 노드에 붙은 관계까지 함께 지운다

# 3) 벡터·전문 인덱스: 노드를 지워도 인덱스는 남는다. 차원이 다른 옛 인덱스가 남아 있으면 뒤에서 걸린다
for _ix in run_cypher("SHOW INDEXES YIELD name, type WHERE type IN ['VECTOR','FULLTEXT'] RETURN name"):
    run_cypher(f"DROP INDEX {_ix['name']} IF EXISTS")   # IF EXISTS: 이미 없어도 에러 없이 넘어간다

print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

아래 두 셀이 지난 시간에 만든 상태를 그대로 다시 만듭니다. 이 노트북 혼자 실행해도 되도록 **그래프 적재 · 논문 적재 · 임베딩 · 벡터/전문 인덱스 · `MENTIONS`** 까지 한 번에 준비합니다.

In [ ]:
# [제공 코드] 의료 지식 그래프 적재: 이 셀은 실행만 하세요(2초쯤 걸립니다).
# 32일차에서 적재한 그 그래프입니다(Hetionet v1.0 중 재배포 가능한 CC0 부분, 2016년 자료).
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")   # 정답 폴더에서도 돌게
# 이 그래프의 노드 레이블 5종: 약물·질병·유전자·증상·약효분류
NODE_LABELS = ['Compound', 'Disease', 'Gene', 'Symptom', 'PharmacologicClass']
# 관계 이름 -> (출발 레이블, 도착 레이블). 아래 MATCH 에 레이블을 찍어 인덱스를 타게 하려고 미리 적어 둔다
REL_ENDS = {
    "TREATS": ("Compound", "Disease"),
    "PALLIATES": ("Compound", "Disease"),
    "BINDS": ("Compound", "Gene"),
    "UPREGULATES_CG": ("Compound", "Gene"),
    "DOWNREGULATES_CG": ("Compound", "Gene"),
    "ASSOCIATES": ("Disease", "Gene"),
    "UPREGULATES_DG": ("Disease", "Gene"),
    "DOWNREGULATES_DG": ("Disease", "Gene"),
    "RESEMBLES_CC": ("Compound", "Compound"),
    "RESEMBLES_DD": ("Disease", "Disease"),
    "PRESENTS": ("Disease", "Symptom"),
    "INCLUDES": ("PharmacologicClass", "Compound"),
}

# 1) id 인덱스부터: 관계를 만들 때 노드를 id 로 찾으므로, 없으면 매번 전수 스캔이 된다
for _label in NODE_LABELS:
    run_cypher(f"CREATE INDEX {_label.lower()}_id IF NOT EXISTS FOR (n:{_label}) ON (n.id)")

# 2) 노드 적재: csv 를 레이블별로 나눠 담고 레이블마다 한 번에 보낸다
_nodes = {_label: [] for _label in NODE_LABELS}
for _row in pd.read_csv(DATA_DIR / "hetionet_nodes.csv").to_dict("records"):
    _nodes[_row["label"]].append({"id": _row["id"], "name": _row["name"]})
for _label, _rows in _nodes.items():
    run_cypher(f"UNWIND $rows AS row CREATE (n:{_label}) SET n.id = row.id, n.name = row.name",
               rows=_rows)

# 3) 관계 적재: 2만 건씩 끊어 보낸다(한 번에 다 보내면 메모리를 많이 쓴다)
_edges = {_rel: [] for _rel in REL_ENDS}
for _row in pd.read_csv(DATA_DIR / "hetionet_edges.csv").to_dict("records"):
    _edges[_row["rel"]].append({"s": _row["source"], "t": _row["target"]})
for _rel, _rows in _edges.items():
    _src, _dst = REL_ENDS[_rel]
    for _start in range(0, len(_rows), 20000):
        run_cypher(f"UNWIND $rows AS row "
                   f"MATCH (a:{_src} {{id: row.s}}), (b:{_dst} {{id: row.t}}) "
                   f"CREATE (a)-[:{_rel}]->(b)", rows=_rows[_start:_start + 20000])

print("노드:", run_cypher("MATCH (n) RETURN count(n) AS c")[0]["c"],
      "/ 관계:", run_cypher("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"])

In [ ]:
# [제공 코드] 검색 준비 완료 상태 만들기: 적재 -> 임베딩 -> 벡터/전문 인덱스 -> MENTIONS (실행만 하세요).
import gzip
import json
import re
from pathlib import Path

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
# 1) 논문 69편을 읽어 Document 노드로 올린다
papers = [json.loads(_line) for _line in
          (DATA_DIR / "pmc_docs.jsonl").read_text(encoding="utf-8").splitlines() if _line.strip()]
run_cypher("CREATE INDEX document_pmcid IF NOT EXISTS FOR (n:Document) ON (n.pmcid)")
run_cypher("UNWIND $rows AS row CREATE (n:Document) SET n += row", rows=papers)

# 2) 제목과 본문을 이어 임베딩하고 노드 속성 emb 에 넣는다(지난 시간에 직접 해 본 그것)
_vectors = embed_texts([_p["title"] + " " + _p["text"] for _p in papers])
run_cypher("""UNWIND $rows AS row
              MATCH (n:Document {pmcid: row.pmcid})
              CALL db.create.setNodeVectorProperty(n, 'emb', row.vec)""",
           rows=[{"pmcid": _p["pmcid"], "vec": _vec} for _p, _vec in zip(papers, _vectors)])
# 3) 인덱스 둘: 뜻으로 찾는 벡터 인덱스(차원 768·코사인은 임베딩과 맞춘다)와 글자로 찾는 전문 인덱스.
run_cypher("""CREATE VECTOR INDEX doc_vec IF NOT EXISTS FOR (n:Document) ON n.emb
             OPTIONS {indexConfig: {`vector.dimensions`: 768,
                                    `vector.similarity_function`: 'cosine'}}""")
run_cypher("CREATE FULLTEXT INDEX doc_ft IF NOT EXISTS FOR (n:Document) ON EACH [n.title, n.text]")
run_cypher("CALL db.awaitIndexes()")

# 4) 이름 사전을 열어 논문에서 개체 이름을 찾을 준비를 한다
with gzip.open(DATA_DIR / "name2id.json.gz", "rt", encoding="utf-8") as _f:
    NAME2ID = json.load(_f)

TOKEN = re.compile(r"[A-Za-z][A-Za-z0-9'-]*")   # 한 단어의 모양: 영문으로 시작하고 숫자·따옴표·하이픈까지 한 단어로 본다


def find_entities(text, name2id):
    """문서에서 사전에 있는 이름을 찾아 {id: (레이블, 표준 이름)} 으로 돌려줍니다."""
    entries, genes = name2id["entries"], name2id["genes"]
    ambiguous, brand_stopwords = name2id["ambiguous"], name2id["brand_stopwords"]
    found = {}   # 키가 노드 id 라 같은 개체를 두 번 잡아도 한 번만 남는다
    for word in TOKEN.findall(text):
        # 유전자 기호는 대소문자를 그대로 맞춥니다(소문자로 누르면 평범한 단어가 유전자가 됩니다)
        if word in genes:
            found[genes[word]] = ("Gene", word)
            continue
        name = word.lower()   # 나머지 이름은 소문자로 맞춰 사전을 찾습니다
        # ambiguous: 종류가 다른 노드 둘에 걸리는 이름 9개. 잘못 합칠 바에는 안 잇습니다
        if name not in entries or name in ambiguous:
            continue
        # 흔한 영어 단어와 겹치는 상품명은 원래 대소문자로 쓰인 자리만 인정합니다
        if name in brand_stopwords and word.islower():
            continue
        entry = entries[name]
        found[entry["id"]] = (entry["label"], entry["canonical"])
    return found


# 5) 논문마다 개체를 찾아 MENTIONS 로 잇는다. 검색이 그래프로 넘어가는 다리다
_links = {}
for _p in papers:
    for _eid, (_label, _canon) in find_entities(_p["title"] + " " + _p["text"], NAME2ID).items():
        _links.setdefault(_label, []).append({"pmcid": _p["pmcid"], "id": _eid})
for _label, _rows in _links.items():
    run_cypher(f"UNWIND $rows AS row MATCH (d:Document {{pmcid: row.pmcid}}), "
               f"(e:{_label} {{id: row.id}}) MERGE (d)-[:MENTIONS]->(e)", rows=_rows)

print("준비 완료: 논문", len(papers), "편 · MENTIONS",
      run_cypher("MATCH (:Document)-[r:MENTIONS]->() RETURN count(r) AS c")[0]["c"], "건")

---
# 1. 검색기 조립: VectorRetriever

> **기본 GraphRAG 의 첫 걸음입니다.** 질문으로 근거가 될 문서를 찾는 자리입니다.

손으로 짜던 검색을 라이브러리 부품으로 바꿔 조립합니다. 그러고 나서 **아직 안 쓴 인덱스 하나**를 같이 걸어 봅니다.

- **1-1** `VectorRetriever` 로 의미 검색을 조립하고 결과의 모양을 읽습니다.
- **1-2** 같은 기호를 벡터·전문 두 인덱스에 나란히 걸어 무엇이 갈리는지 봅니다.

## 1-1. `VectorRetriever` 로 의미 검색 조립하기

### 왜 검색기를 쓸까요?
지난 시간엔 **질문 임베딩 -> `SEARCH` 쿼리 -> 결과 정리**를 매번 손으로 이었습니다. **neo4j-graphrag** 의 `VectorRetriever` 는 이 과정을 하나로 묶어, **질문 문장만 주면** 검색 결과를 일정한 형식으로 돌려줍니다. 이렇게 "질문을 받아 관련 문서를 꺼내 오는 부품"을 **리트리버(retriever)** 라고 부릅니다.

### 준비물: 임베더 어댑터
검색기는 질문을 **스스로 임베딩**하므로 임베더를 넘겨줍니다. 라이브러리가 주는 것을 그대로 쓰는 것이 보통이고 `neo4j_graphrag.embeddings` 에 OpenAI·Gemini·Ollama 등이 들어 있습니다. 그런데도 여기서 한 줄짜리 어댑터를 만드는 이유는 하나입니다. **`dimensions=768` 을 고정하려고요.** 라이브러리의 `OpenAIEmbeddings` 는 생성자에 준 값을 OpenAI 클라이언트로 넘기고, 검색기는 `embed_query(질문)` 을 **인자 없이** 부릅니다. 768을 끼워 넣을 자리가 없다는 뜻입니다. 문서를 768차원으로 적재해 두었으니 질문도 같은 768차원이어야 같은 공간에서 잽니다.

> **질문은 저장하지 않습니다.** 준비 셀의 `embed_texts` 는 논문 69편을 다시 부르지 않으려고 파일에 저장해 두지만, `embed_query` 는 저장하지 않습니다. 표준 도구도 그렇습니다. LangChain 의 `CacheBackedEmbeddings` 는 `query_embedding_cache` 기본값이 `False` 라 **문서만** 저장합니다. 질문은 매번 달라져 다시 쓸 일이 없고, 한 문장 임베딩은 값이 거의 들지 않습니다.

In [ ]:
# [제공 코드] 검색기용 임베더 어댑터: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
from neo4j_graphrag.embeddings.base import Embedder


class QueryEmbedder(Embedder):
    """검색기가 요구하는 embed_query 자리에 준비 셀의 embedder 를 끼워 넣는 어댑터."""

    def embed_query(self, text):
        """질문 한 문장을 문서와 같은 768차원으로."""
        return embedder.embed_query(text)   # 준비 셀에서 dimensions=768 로 만들어 둔 그 embedder


query_embedder = QueryEmbedder()

print("임베더 어댑터 준비 완료")

이제 검색기를 만듭니다. `VectorRetriever(driver, '인덱스명', embedder=..., return_properties=[...])` 이고, `return_properties` 는 결과에 담아 돌려줄 노드 속성입니다. **여기서는 `pmcid` 와 `title` 만 적었으니 논문 본문은 오지 않습니다.**

> **왜 본문을 빼나요?** 오늘 순위를 정하는 신호(유사도, 2-2 에서 새길 그래프 점수)는 **본문을 읽지 않습니다.** 그래서 본문은 4-1 에서 세 편으로 추린 뒤 그 세 편만 꺼내면 됩니다. 검색할 때 다 받으면 상위 20편에서 9만 자가 넘습니다. **순위 신호가 본문을 읽어야 하는 방식이라면 이야기가 달라집니다.**

> **`return_properties` 는 노드 자기 속성만 고릅니다.** `text` 처럼 노드에 적힌 것은 여기 적으면 오지만, 4절에서 근거에 실을 **개체 목록**은 `MENTIONS` 를 한 걸음 건너야 나오는 것이라 이 목록에 자리가 없습니다. 그건 4-1 에서 Cypher 로 따로 가져옵니다.

<img src="images/VectorRetriever_부품.png" alt="질문 문장이 임베더, 벡터 인덱스, 포맷터를 거쳐 결과 항목이 되는 흐름" width="900">

In [ ]:
# neo4j-graphrag 의 VectorRetriever 로 '질문 -> 임베딩 -> 벡터 검색'을 한 줄로 묶습니다.
from neo4j_graphrag.retrievers import VectorRetriever

# 'doc_vec' 은 교안_01 3-1 에서 우리가 지은 인덱스 이름입니다. return_properties: 결과에 담아 돌려줄 노드 속성
retriever = VectorRetriever(driver, 'doc_vec', embedder=query_embedder,
                            return_properties=['pmcid', 'title'])

question = '어떤 유전자가 약물 대사에 관여하나요?'
# top_k 는 '최대 몇 편'이라는 예산이다. 1-2 에서 이 예산이 무엇을 놓치게 하는지 본다
result = retriever.search(query_text=question, top_k=3)   # 질문 임베딩은 검색기가 query_embedder 로 한다

# 돌려받은 것을 먼저 그대로 찍어 봅니다. 무엇이 오는지 알고 나서 꺼내 씁니다
print(type(result).__name__, '· 항목', len(result.items), '개')

In [ ]:
print(result.items[0])

In [ ]:
print('content 의 자료형:', type(result.items[0].content).__name__)

`RetrieverResult` 안에 `items` 가 있고, 항목 하나는 `content` 와 `metadata` 를 갖습니다. `metadata` 에는 우리가 안 시킨 `nodeLabels`·`id` 까지 들어 있습니다. 손으로 짜던 검색이 **한 번의 `search` 호출**로 정리됐습니다.

> 다만 `content` 는 **문자열**입니다. 딕셔너리처럼 보이지만 대괄호로 꺼낼 수 없습니다. 결과의 모양을 우리가 정하려면 **`result_formatter`** 를 넘깁니다. 검색 결과 한 행(`record`)을 받아 `RetrieverResultItem(content=..., metadata=...)` 로 바꿔 주는 함수입니다.

In [ ]:
# 결과 한 행을 우리가 원하는 모양으로 바꾼다. record['node'] 가 노드, record['score'] 가 점수다
from neo4j_graphrag.types import RetrieverResultItem


def to_item(record):
    """본문 발췌는 content 에, pmcid 와 제목과 점수는 metadata 에 담아 돌려준다."""
    paper = record['node']   # return_properties 를 안 주면 노드 속성이 통째로 온다(본문 text 까지)
    return RetrieverResultItem(content=paper['text'][:120],
                               metadata={'pmcid': paper['pmcid'], 'title': paper['title'],
                                         'score': record['score']})


# 같은 검색기를 result_formatter 를 달아 다시 만든다. 검색은 같고 결과의 모양만 달라진다
# return_properties 를 안 줬으므로 이번엔 본문 text 도 함께 온다(그래서 발췌를 뜰 수 있다)
retriever = VectorRetriever(driver, 'doc_vec', embedder=query_embedder, result_formatter=to_item)
result = retriever.search(query_text=question, top_k=3)
for item in result.items:
    # 이제 pmcid 는 metadata 에서 곧장 꺼낸다. 문자열을 가를 필요가 없다
    print(item.metadata['pmcid'], round(item.metadata['score'], 3), item.content[:50])

이제 `pmcid` 와 점수를 `metadata` 에서 곧장 꺼냅니다. 2절부터는 **우리가 직접 쓴 `SEARCH` 쿼리**로 갑니다. 검색기로도 같은 것을 받을 수 있지만, 2-3 의 `rerank` 가 유사도와 그래프 점수를 **한 행의 딕셔너리**로 받아야 해서 그 모양을 바로 주는 쪽이 짧습니다. 두 방법의 검색 결과가 같은지 먼저 확인합니다.

In [ ]:
# 같은 검색을 우리가 직접 쓴 SEARCH 쿼리로 해 봅니다. 결과가 같은지 확인합니다.
def search_docs(question, top_k=5):
    """질문으로 논문을 찾아 pmcid·제목·유사도·그래프 점수를 함께 돌려준다."""
    # graph_score 는 아직 없는 속성이라 지금은 None 으로 온다. 2-2 에서 채운다
    return run_cypher('''MATCH (n:Document)
                           SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT $k) SCORE AS score
                         RETURN n.pmcid AS pmcid, n.title AS title, score,
                                n.graph_score AS graph_score
                         ORDER BY score DESC''',
                      q=embed_query(question), k=top_k)


# result_formatter 덕에 pmcid 를 metadata 에서 바로 꺼낸다
print('검색기      :', [item.metadata['pmcid'] for item in result.items])
print('직접 쓴 쿼리:', [hit['pmcid'] for hit in search_docs(question, top_k=3)])

> 검색기가 속으로 보내는 질의도 **우리가 배운 `SEARCH` 절**입니다. 서버 버전을 보고 고르는데, 2026.01 이상이면 `SEARCH` 를 씁니다. 결과가 같은 것은 같은 인덱스를 같은 문법으로 두드리기 때문입니다.

> **다만 `search(..., filters={'year': '2025'})` 는 사전 필터가 아닙니다.** 우리 `doc_vec` 은 거를 속성을 등록해 두지 않았기 때문입니다(교안_01 3-2 의 `WITH [n.year]`). 그래서 라이브러리가 **"filterable 로 선언되지 않았다"는 경고를 찍고** 인덱스를 건너뛴 채 조건에 맞는 문서를 전부 `vector.similarity.cosine` 으로 잽니다. 교안_01 3-1 의 전수 비교입니다.

### 🖐️ 함께 따라하기: 다른 질문으로 검색기 써 보기

같은 검색기에 **다른 질문**을 넣어 봅니다.

1. 질문 **'약물 상호작용은 어디에서 일어나나요?'** 로 `top_k=3` 검색하세요.
2. 각 항목의 유사도(소수 셋째 자리까지)와 `content` 를 출력하세요.
3. 같은 질문을 `search_docs` 로도 돌려 `pmcid` 목록이 같은지 확인하세요.

**확인 기준**: 두 방법이 같은 세 편을 같은 순서로 돌려줍니다. 1위는 리팜핀과 플루바스타틴의 상호작용을 다룬 논문이 아니라 프로프라놀롤의 β-수용체 비의존 작용을 다룬 `PMC13467852` 입니다. 뜻으로 매긴 1위가 늘 질문에 딱 맞는 논문은 아니라는 것도 함께 보세요.

In [ ]:
# 여기에 코드를 작성하세요

## 1-2. 인덱스는 둘인데 아직 하나만 썼습니다

### 왜 지금 보나요?
교안_01 4-2 에서 같은 기호로 두 인덱스를 견줬습니다. 여기서 새로 보는 것은 하나입니다. **아예 못 찾는 것과, 같은 예산(`top_k`) 안에서 밀리는 것은 다릅니다.**

In [ ]:
# 준비 셀은 벡터 인덱스 doc_vec 말고 전문 인덱스 doc_ft 도 만들어 두었습니다. 같은 기호를 둘에 나란히 겁니다.
symbol = 'CYP2C19'   # 데모의 기준 약(클로피도그렐)을 활성형으로 바꾸는 대사 효소 기호
by_vector = [hit['pmcid'] for hit in search_docs(symbol, top_k=5)]
# 전문 인덱스는 아직 프로시저로 부릅니다. 전문 인덱스에는 SEARCH 절이 없습니다
by_term = [row['pmcid'] for row in run_cypher(
    '''CALL db.index.fulltext.queryNodes('doc_ft', $t) YIELD node, score
       RETURN node.pmcid AS pmcid ORDER BY score DESC''', t=symbol)]
print('의미 검색 상위 5:', by_vector)
print('키워드 검색      :', by_term)

In [ ]:
# 한쪽만 찾은 논문이 곧 '한 인덱스만 쓰면 놓치는 것'이다
only_term = [pmcid for pmcid in by_term if pmcid not in by_vector]
print('키워드 검색만 찾은 논문:', only_term)

In [ ]:
# 후보 예산(top_k)을 올려 다시 받아 본다. 아예 못 찾는 것과 같은 예산 안에서 밀리는 것은 다르다
wider = [hit['pmcid'] for hit in search_docs(symbol, top_k=10)]
print('그중 top_k 를 10 으로 올려도 못 본 것:',
      [pmcid for pmcid in only_term if pmcid not in wider])

In [ ]:
# 합집합이 곧 두 인덱스를 다 쓸 때의 커버 범위
print('둘을 합쳐 받으면     :', len(set(by_vector) | set(by_term)), '편')

키워드 검색만 찾은 논문이 **1편** 있습니다. 벡터 인덱스가 이 논문을 **못 찾는 것이 아니라**, 같은 예산(`top_k=5`) 안에서 **못 보는 것**입니다. 뜻으로 매긴 순위에서 15위라 `top_k` 를 15 로 올려야 들어옵니다. 키워드 검색은 그 키워드가 든 것을 **예산과 상관없이** 모으니까요. 두 인덱스를 다 두는 이유가 이것입니다.

> 다만 그냥 이어 붙일 수는 없습니다. 유사도와 키워드 점수는 **눈금이 다르기** 때문입니다(2-3 에서 같은 문제를 다시 만납니다). 두 순위를 제대로 섞는 방법은 **47일차의 하이브리드 검색**에서 다룹니다. 오늘은 여기까지만 확인하고, 이 노트북의 나머지는 `doc_vec` 으로 갑니다.

> 두 인덱스를 함께 두드리는 부품은 라이브러리에도 있습니다(`HybridRetriever`). **47일차**에서 씁니다. 오늘은 그 안에서 무슨 일이 벌어지는지를 손으로 조립해 봅니다.

### ✅ 바로 확인 퀴즈 (1-1·1-2)

**1.** `VectorRetriever` 에 임베더(`query_embedder`)를 넘기는 이유는? 그리고 라이브러리가 주는 임베더를 그대로 쓰지 않고 한 줄짜리 어댑터를 만든 이유는?

<details><summary>정답 보기</summary>

검색기가 **질문 문장을 스스로 임베딩**해 벡터 검색을 하기 때문입니다. 어댑터를 만든 것은 **`dimensions=768` 을 고정하려고**입니다. 검색기는 `embed_query(질문)` 을 인자 없이 부르는데, 라이브러리 임베더는 생성자에 준 값을 OpenAI 클라이언트로 넘겨서 768을 끼워 넣을 자리가 없습니다. 문서가 768차원이니 질문도 768차원이어야 같은 공간에서 잽니다.

</details>

**2.** `return_properties` 에 `title` 만 넣고 검색해 그 결과를 근거로 넘겼습니다. 답변이 부실한 이유는 무엇일까요?

<details><summary>정답 보기</summary>

`content` 에 **제목만** 담겨 근거에 알맹이가 없기 때문입니다. 검색기 결과를 그대로 근거로 쓰는 구조라면 본문을 `return_properties` 에 넣어야 하고, 오늘처럼 고른 뒤 따로 꺼내는 구조라면 4-1 처럼 `pmcid` 로 다시 조회해야 합니다. 어느 쪽이든 **근거에는 본문이 실려야** 합니다.

</details>

**3.** 같은 기호 `CYP2C19` 를 벡터·전문 두 인덱스에 걸었더니 **키워드 검색만 찾은 논문이 1편** 있었습니다. 벡터 검색이 이 논문을 놓친 까닭은?

<details><summary>정답 보기</summary>

벡터는 **뜻**으로 상위 몇 편만 채워 옵니다. 그 기호가 적혀 있어도 문장 전체의 뜻이 질문과 덜 가까우면 순위 밖으로 밀립니다. "그 기호가 든 것을 **빠짐없이**" 모으는 일은 키워드 검색의 몫입니다.

</details>

---
# 2. PageRank 를 검색 순위에 얹기

> **여기부터 두 절이 검색 최적화입니다.** 기본 GraphRAG 는 1절의 검색만으로도 돌아갑니다. 2·3절은 그 검색이 고른 후보를 손보는 도구를 만들고, 4절에서 얹어 봅니다.

검색이 고른 후보의 **순서를 그래프 구조로 다듬습니다.** 세 소단원이 한 줄로 이어집니다.

- **2-1** 투영을 다시 만들고 PageRank 를 개체에 새깁니다.
- **2-2** 개체 점수를 문서 점수로 옮깁니다.
- **2-3** 눈금을 맞춰 두 점수를 섞고, 비율이 순위를 어떻게 바꾸는지 잽니다.

## 2-1. 투영과 PageRank: 개체에 점수 새기기

### 왜 필요할까요?
의미만으로 고르면 **뜻은 가깝지만 그래프에서는 변두리인** 논문이 위로 올 수 있습니다. 우리에게는 지식그래프가 있으니, 그 논문이 다루는 개체가 **그래프에서 얼마나 중심에 있는지**를 함께 볼 수 있습니다.

> 한 가지 먼저 짚습니다. 지금부터 만드는 그래프 점수는 **질문이 오기 전에 한 번 계산해 노드에 새겨 두는 값**입니다. 질문마다 다시 재지 않습니다. 그래서 이 점수는 **어떤 질문에서든 같은 논문을 같은 방향으로 밀어 올립니다.** 2-3 에서 섞는 비율을 너무 높이면 질문이 무엇이든 순위가 비슷해지는 까닭이 이것입니다.

### 세 걸음으로 합니다
1. **33일차**의 **투영**을 다시 만든다(약물 중심). 원본 그래프에서 약물 둘레만 잘라 온 메모리 위 사본입니다.
2. **PageRank 를 개체 노드에 새긴다**(`write`). 문서가 아니라 개체 쪽입니다.
3. 그 점수를 **문서로 옮긴다**: 그 문서가 언급한 개체들의 PageRank **평균**(2-2 에서 합니다).

<img src="images/GDS투영과_PageRank.png" alt="원본 그래프에서 약물 둘레만 투영해 PageRank 를 재고 개체 노드에 새기는 흐름" width="900">

In [ ]:
# 33일차에서 중심성을 재던 그 투영(약물 중심)을 다시 만듭니다. 적은 레이블·관계만 골라 메모리 위 사본으로 뜹니다.
# UNDIRECTED 는 관계 하나를 양방향 두 건으로 싣습니다. 그래서 찍히는 관계 수가 DB 의 두 배가 됩니다
run_cypher("""
    CALL gds.graph.project('drugGraph',
        ['Compound', 'Disease', 'PharmacologicClass'],
        {
        TREATS: {orientation: 'UNDIRECTED'},
         PALLIATES: {orientation: 'UNDIRECTED'},
         INCLUDES: {orientation: 'UNDIRECTED'},
         RESEMBLES_DD: {orientation: 'UNDIRECTED'},
         RESEMBLES_CC: {orientation: 'UNDIRECTED'}})
    YIELD graphName RETURN graphName
""")
info = run_cypher("CALL gds.graph.list('drugGraph') "
                  "YIELD nodeCount, relationshipCount RETURN nodeCount, relationshipCount")[0]
print(f"투영 노드 {info['nodeCount']:,} / 관계 {info['relationshipCount']:,}")

In [ ]:
# PageRank 를 계산해 개체 노드의 pagerank 속성에 새깁니다(문서가 아니라 개체 쪽입니다).
run_cypher("CALL gds.pageRank.write('drugGraph', {writeProperty:'pagerank'}) "
           "YIELD nodePropertiesWritten RETURN nodePropertiesWritten")
for row in run_cypher('MATCH (n) WHERE n.pagerank IS NOT NULL '
                      'RETURN n.name AS name, labels(n)[0] AS label, '
                      'round(n.pagerank, 2) AS pr ORDER BY pr DESC LIMIT 5'):
    print(f"  {row['name']:26} {row['label']:12} {row['pr']}")

In [ ]:
# 이 최솟값은 이웃이 하나도 없는 노드가 받는 값이라 그래프 전체의 바닥값이 된다
print('가장 낮은 값:', run_cypher('MATCH (n) WHERE n.pagerank IS NOT NULL '
                                'RETURN round(min(n.pagerank), 2) AS lo')[0]['lo'])

질병들이 위에 올라옵니다. 이 투영에서는 하나의 질병에 여러 약이 붙기 때문입니다. 가장 낮은 값이 **0.15** 인 것도 눈여겨보세요. 33일차에서 본 그 바닥값이고, 2-2 에서 이 값을 씁니다.

## 2-2. 개체 점수를 문서 점수로 옮기기

### 왜 평균인가
최댓값을 쓰면 논문이 `hypertension` 을 한 번 스치듯 언급하기만 해도 그래프에서 가장 중심인 노드의 점수를 그대로 받습니다. 평균은 "이 논문이 다루는 개체들이 **대체로** 얼마나 중심에 있나"를 봅니다.

> **다만 이 투영에는 유전자·증상이 없습니다.** 약물·질병·약효분류만 담았으므로 유전자와 증상은 `pagerank` 가 아예 없고, 아래 `WHERE e.pagerank IS NOT NULL` 이 그것들을 **평균의 분모에서 통째로 뺍니다.** 그래서 개체를 여럿 언급했더라도 **투영에 든 개체가 하나뿐인 논문은 평균이라도 그 개체의 점수를 그대로 받습니다.** 실제로 `graph_score` 1위는 `PMC13489523` 과 `PMC13494715` 두 편이 똑같은 값으로 나란히 섭니다. 개체를 6개와 14개로 다르게 언급했는데도 투영에 든 것이 둘 다 `hypertension` 하나뿐이라 같은 점수를 받은 것입니다. 평균이 최댓값을 완전히 막아 주지는 않는다는 뜻입니다.

<img src="images/개체점수를_문서점수로.png" alt="논문이 언급한 개체 가운데 투영에 든 것의 PageRank 평균이 문서 점수가 된다" width="900">

In [ ]:
# 투영에 없는 개체(유전자·증상)는 pagerank 가 없어 분모에서 통째로 빠진다
run_cypher('''MATCH (d:Document)-[:MENTIONS]->(e) WHERE e.pagerank IS NOT NULL
              WITH d, avg(e.pagerank) AS mean SET d.graph_score = mean''')
# 투영에 든 개체를 하나도 안 쓴 논문에는 바닥값 0.15 를 줍니다(0 을 주면 외톨이 개체보다 낮은 자리를 새로 만드는 셈입니다).
run_cypher('MATCH (d:Document) WHERE d.graph_score IS NULL SET d.graph_score = 0.15')
print('개체에서 점수를 받은 논문:',
      run_cypher('MATCH (d:Document) WHERE d.graph_score > 0.15 '
                 'RETURN count(d) AS c')[0]['c'], '편 / 69편')

69편 중 58편이 개체에서 점수를 받았습니다. 나머지 11편은 이 투영에 든 개체(약물·질병·약효분류)를 하나도 쓰지 않은 논문입니다. 대개 **유전자 이야기만 하는 논문**이죠. 그 논문들에는 바닥값을 줍니다.

## 2-3. 두 점수를 섞을 때 눈금을 맞춘다

검색 점수는 0.70~0.73 처럼 좁게 몰려 있고(폭 0.028), 그래프 점수는 0.15에서 8까지 벌어집니다. 그냥 곱하거나 더하면 **그래프 점수가 순위를 다 정해 버립니다.** 후보 안에서 0에서 1 사이로 눌러 맞춘 뒤 섞습니다.

> 검색 점수의 폭이 좁은 까닭은 교안_01 3-2 에서 봤습니다. 이 모델의 코사인 자체가 좁은 구간에 몰려 있고, **`(1+코사인)/2`** 로 옮기면서 그 폭이 다시 절반으로 눌립니다.

In [ ]:
# 눌러 맞추기는 질의마다 이 후보 안에서 새로 한다. 전체 문서 기준으로 하면 후보 간 차이가 뭉개진다
def minmax(values):
    """후보 안에서 가장 낮은 값을 0, 가장 높은 값을 1 로 눌러 맞춘다."""
    low, high = min(values), max(values)
    # 후보가 모두 같은 값이면 나눌 수가 없습니다. 그때는 가운데 값으로 둡니다
    return [0.5 if high == low else (v - low) / (high - low) for v in values]


# 폭이 열여덟 배 차이 나는 두 줄을 같은 자로 눌러 봅니다
similarity = [0.703, 0.706, 0.708, 0.731]   # 폭 0.028
graph = [0.80, 0.95, 1.11, 1.31]            # 폭 0.51
print('유사도   ', similarity, '->', [round(v, 2) for v in minmax(similarity)])
print('그래프   ', graph, '->', [round(v, 2) for v in minmax(graph)])
print('모두 같으면', [1.0, 1.0, 1.0], '->', minmax([1.0, 1.0, 1.0]))

폭이 0.028 인 유사도와 0.51 인 그래프 점수가 **둘 다 0에서 1 사이**가 됐습니다. 이제 같은 자로 잰 값이라 섞을 수 있습니다. 셋째 줄은 후보가 모두 같은 값일 때인데, 그때는 나눌 수가 없어 가운데인 0.5 로 둡니다.

In [ ]:
# 섞기: 눌러 맞춘 두 점수를 weight 비율로 더해 다시 정렬합니다.
def rerank(hits, weight):
    """유사도와 그래프 점수를 weight 비율로 섞어 다시 정렬한 pmcid 목록."""
    sim = minmax([hit['score'] for hit in hits])            # 유사도를 0~1 로
    graph = minmax([hit['graph_score'] for hit in hits])    # 그래프 점수도 같은 자로
    # weight 가 0 이면 유사도만, 1 이면 그래프 점수만. 그 사이에서 두 순위를 섞는다
    fused = [(1 - weight) * s + weight * g for s, g in zip(sim, graph)]
    return [hit['pmcid'] for hit, _ in sorted(zip(hits, fused), key=lambda pair: -pair[1])]


# 두 편만으로 손계산해 봅니다. A 는 유사도가 높고 B 는 그래프 점수가 높습니다
example = [{'pmcid': 'A', 'score': 0.73, 'graph_score': 0.8},
           {'pmcid': 'B', 'score': 0.70, 'graph_score': 1.3}]
for w in (0.0, 0.3, 0.5, 0.7, 1.0):
    print(f'  w={w}: {rerank(example, w)}')

두 편뿐이라 눌러 맞추면 유사도는 `[1, 0]`, 그래프 점수는 `[0, 1]` 이 됩니다. 섞은 값은 `[1-w, w]` 가 되죠. 그래서 **`w` 가 0.5 보다 작으면 A, 크면 B** 가 1위입니다. `w` 는 "그래프 점수를 얼마나 믿을 것인가" 를 정하는 손잡이입니다.

In [ ]:
# 이제 진짜 후보 5편에 걸어 봅니다. 눌러 맞추기는 이 다섯 편 안에서만 합니다
hits = search_docs(question, top_k=5)
for hit in hits:
    print(f"  {hit['pmcid']}  유사도 {hit['score']:.3f}  그래프 {hit['graph_score']:.2f}  "
          f"{hit['title'][:44]}")

In [ ]:
print('의미만       :', [hit['pmcid'] for hit in hits])
print('섞기 w=0.3   :', rerank(hits, 0.3))
print('섞기 w=0.7   :', rerank(hits, 0.7))

**`w=0.3` 에서 벌써 순위가 크게 움직입니다.** 5위였던 `PMC13464512` 가 2위로 올라옵니다. 타목시펜 논문인데 그래프 점수(1.31)가 다섯 편 중 가장 높기 때문입니다. 유사도 폭이 0.028 밖에 안 돼, 눌러 맞추고 나면 그래프 점수 차이가 그대로 순위를 흔듭니다.

**`w=0.7` 로 올리면 1위까지 차지합니다.** 그런데 `PMC13464512` 는 타목시펜 논문입니다. "어떤 유전자가 약물 대사에 관여하나"라는 질문에 **타목시펜 논문이 1위**인 것은 좋은 답이 아닙니다. 그래프 점수를 너무 믿으면 **질문에서 멀어집니다.**

> 섞는 비율에는 적당한 값이 있습니다. 어느 값이 맞는지는 **재 봐야 압니다.** 정답이 있는 질문 몇 개를 골라 두고 비율을 바꿔 가며 순위를 확인하는 것이 실무에서 하는 일입니다.

<img src="images/리랭킹_순위역전.png" alt="검색 점수만 본 순위와 그래프 점수를 섞은 순위의 비교" width="900">

그림의 막대 라벨은 `pmcid` 의 뒤 네 자리입니다(4111 = PMC13494111). 왼쪽은 검색 점수만 본 순위로 다섯 편이 0.028 폭 안에 몰려 있고, 오른쪽은 `w=0.7` 로 섞은 순위입니다. 그래프 점수는 0.80 에서 1.31 까지 벌어지므로, 크게 섞으면 질문과 덜 맞는 타목시펜 논문(4512)이 1위로 올라옵니다.

### 🖐️ 함께 따라하기: 다른 질문으로 비율 바꿔 보기

1-1 의 따라하기에서 쓴 질문('약물 상호작용은 어디에서 일어나나요?')으로 같은 일을 해 봅니다.

1. `search_docs` 로 **상위 5편**을 받으세요.
2. `pmcid`·유사도·그래프 점수를 한 줄씩 출력하세요.
3. `rerank` 를 **`w=0.3`** 과 **`w=0.7`** 로 각각 돌려 의미만의 순위와 나란히 출력하세요.

**확인 기준**: 이 질문에서는 **`w=0.3` 에서도 `w=0.7` 에서도 앞 세 편이 그대로**입니다. `w=0.7` 에서 4위와 5위만 자리를 바꿉니다. 그래프 점수가 0.77 에서 1.37 사이라 서로 크게 다르지 않기 때문입니다. **섞는다고 늘 순위가 뒤집히는 것은 아닙니다.** 데모 질문에서 크게 흔들린 것은 그 후보들의 그래프 점수 차이가 컸기 때문입니다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈 (2-2·2-3)

**1.** 두 점수를 섞기 전에 **0에서 1 사이로 눌러 맞추는** 이유는?

<details><summary>정답 보기</summary>

두 점수의 **눈금이 다르기** 때문입니다. 유사도 점수는 `(1+코사인)/2` 로 눌러 맞춘 값이라 0.7 언저리에 몰려 있고, 그래프 점수는 0.15에서 8까지 벌어집니다. 그대로 섞으면 폭이 넓은 쪽이 순위를 다 정해 버립니다.

</details>

**2.** 그래프 점수를 개체 PageRank 의 **최댓값**이 아니라 **평균**으로 만든 이유는?

<details><summary>정답 보기</summary>

최댓값을 쓰면 그래프에서 가장 중심인 개체(`hypertension` 등)를 **한 번 스치듯 언급한 논문**이 그 점수를 통째로 받습니다. 평균은 그 논문이 다루는 개체들이 대체로 얼마나 중심인지를 봅니다.

다만 이 투영에서는 평균이 **완전히 막아 주지는 않습니다.** 유전자·증상은 투영에 없어 `pagerank` 가 없고, 그래서 평균의 분모에서 빠집니다. 투영에 든 개체가 하나뿐인 논문은 평균이라도 그 개체 점수를 그대로 받습니다(`PMC13489523`). 분모를 `MENTIONS` 전체로 넓히면 이 구멍이 막히지만, 그러면 유전자를 많이 언급한 논문일수록 분모만 커져 점수가 낮아집니다.

</details>

**3.** 섞는 비율을 0.7 로 올렸더니 질문과 덜 맞는 논문이 1위가 됐습니다. 무엇을 해야 할까요?

<details><summary>정답 보기</summary>

비율을 **낮춥니다.** 그리고 어느 값이 맞는지는 정답을 아는 질문 몇 개로 **재 봐야** 압니다. 구조 점수는 순위를 다듬는 보조이지, 질문과의 관련성을 대신하지 않습니다.

</details>

---
# 3. 커뮤니티로 검색 범위 좁히기

> **검색 최적화의 둘째 도구입니다.**

리랭킹이 못 하는 일을 합니다. 순서를 흔드는 것이 아니라 **후보를 잘라 내는** 것입니다.

- **3-1** Leiden 으로 개체를 묶고 크기 분포로 읽습니다.
- **3-2** 기준 약이 든 묶음에 무엇이 함께 있는지 들여다봅니다.
- **3-3** 그 묶음으로 검색 범위를 좁힙니다.

<img src="images/리랭킹_vs_커뮤니티필터.png" alt="리랭킹은 후보의 순서를 바꾸고, 커뮤니티 필터는 후보를 잘라 낸다" width="900">

## 3-1. Leiden 으로 묶고 크기 분포 읽기

### 왜 필요할까요?
리랭킹은 순위를 **다듬을** 뿐 엉뚱한 논문을 **빼지는** 못합니다. 2-3 에서 비율을 0.7 로 올리자 타목시펜 논문이 1위까지 올라온 것이 그 예입니다. 비율을 낮추면 1위는 지킬 수 있지만, 그 논문은 여전히 후보 안에 남아 있습니다. 때로는 "이 약과 같은 갈래 안에서만 찾아 달라"가 필요합니다.

**34일차**의 **커뮤니티 탐지**가 촘촘히 이어진 개체들을 하나의 묶음으로 나눠 줍니다. 그 묶음으로 검색 결과를 거르면 주제가 번지지 않습니다.

> **34일차에서 쓴 그 Leiden** 입니다. `randomSeed` 와 `concurrency: 1` 을 함께 줘서 **한 번 올린 데이터 위에서는 몇 번을 돌려도 같은 결과**가 나옵니다(34일차 과제에서 한 그대로입니다). 다만 **데이터를 다시 적재하면** Neo4j 가 노드에 내부 번호를 새로 매기고 Leiden 이 그 순서를 타서 묶음이 달라질 수 있습니다. 그러니 번호를 외워 쓰지 말고 **그때그때 읽어 오는 방법**을 익힙니다.

In [ ]:
# 같은 투영에 커뮤니티 탐지를 돌려 개체마다 묶음 번호를 새깁니다. 관계 구조만 보고 나누므로 번호 자체에는 뜻이 없습니다.
summary = run_cypher("CALL gds.leiden.write('drugGraph', "
                     "{writeProperty:'community', randomSeed: 42, concurrency: 1}) "
                     "YIELD communityCount, modularity "
                     "RETURN communityCount, round(modularity, 3) AS modularity")[0]
print('묶음 수:', summary['communityCount'], '/ 모듈러리티:', summary['modularity'])

In [ ]:
# 묶음 수를 그대로 읽으면 오해합니다. 크기별로 나눠 봅니다
sizes = [row['n'] for row in run_cypher('MATCH (n) WHERE n.community IS NOT NULL '
                                        'RETURN n.community AS c, count(*) AS n '
                                        'ORDER BY n DESC')]
print('혼자인 묶음:', sum(1 for n in sizes if n == 1),
      '/ 2개 이상:', sum(1 for n in sizes if n >= 2),
      '/ 10개 이상:', sum(1 for n in sizes if n >= 10),
      '/ 가장 큰 것:', sizes[0])

> **모듈러리티**는 나눈 결과가 얼마나 또렷한지를 재는 값입니다(-0.5 ~ 1). 0.3 을 넘으면 묶음이 뚜렷하다고 보고, 이 투영은 0.78 이라 잘 나뉜 편입니다. 34일차에서 서로 다른 알고리즘의 분할을 견줄 때 쓴 그 값입니다.

**묶음 수를 그대로 읽으면 오해합니다.** 173개 중 **104개가 혼자짜리**입니다. 이 투영에 관계가 하나도 없는 노드들이죠(약효분류에 안 들고 닮은 약도 없는 화합물). 실제로 쓸 만한 묶음은 10개 이상 모인 **23개**입니다.

## 3-2. 기준 약이 든 묶음 들여다보기

묶음 번호만 봐서는 그 묶음이 무엇의 모임인지 알 수 없습니다. 기준 약이 든 묶음을 열어 무엇이 함께 들어 있는지 봅니다.

In [ ]:
# Clopidogrel 가 어느 묶음에 들었고, 그 묶음에 무엇이 함께 있는지 봅니다.
anchor = 'Clopidogrel'   # 항혈소판제. 이 약이 든 묶음을 기준으로 삼는다
# 묶음 번호 자체에는 뜻이 없다. 외우지 말고 그때그때 읽어 온다
anchor_community = run_cypher('MATCH (a {name:$name}) RETURN a.community AS c',
                              name=anchor)[0]['c']
for row in run_cypher('MATCH (n) WHERE n.community = $c '
                      'RETURN labels(n)[0] AS label, count(*) AS cnt ORDER BY cnt DESC',
                      c=anchor_community):
    print(f"  {row['label']:20} {row['cnt']:>4}")

In [ ]:
# 방금 본 레이블 집계 중 질병만 이름으로 풀어 본다
print('같은 묶음의 질병:', [row['name'] for row in run_cypher(
    'MATCH (n:Disease) WHERE n.community = $c RETURN n.name AS name ORDER BY name',
    c=anchor_community)])

클로피도그렐과 같은 묶음에는 **화합물과 약효분류가 대부분**입니다. 알고리즘이 이름을 보고 나눈 것이 아니라 관계 구조만으로 나눈 결과입니다.

> 함께 든 질병 10개는 한 갈래가 아닙니다. 죽상경화증·관상동맥질환 같은 심혈관 질환과 만성신부전·IgA 사구체신염 같은 신장 질환이 섞여 있습니다. 클로피도그렐이 둘 다에 걸쳐 있기 때문입니다.

> **여러분 화면의 목록은 위와 다를 수 있습니다.** 3-1 에서 말한 대로 데이터를 다시 적재하면 경계에 있는 노드가 이쪽저쪽으로 옮겨 다닙니다.

## 3-3. 그 묶음으로 검색 범위 좁히기

검색 상위 20편을 받아 두고 그중 그 묶음의 개체를 언급한 논문만 남깁니다. 교안_01 3-2 에서 이름을 붙여 둔 **사후 필터**가 이것입니다.

이제 그 묶음을 **거름망**으로 씁니다. 검색 상위 20편을 받아 두고, 그 묶음의 개체를 언급한 논문만 남깁니다.

In [ ]:
# 검색 범위를 그 묶음으로 좁힙니다. 상위 20편을 받아 놓고, 그 묶음의 개체를 언급한 것만 남깁니다.
def search_in_community(question, community, pool=20):
    """벡터 상위 pool 편 중 그 묶음의 개체를 언급한 논문만 순서대로 돌려준다(그래서 pool 보다 적다)."""
    rows = run_cypher('''MATCH (n:Document)
                           SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT $k) SCORE AS score
                         MATCH (n)-[:MENTIONS]->(e) WHERE e.community = $c
                         RETURN DISTINCT n.pmcid AS pmcid, score ORDER BY score DESC''',
                      q=embed_query(question), k=pool, c=community)
    return [row['pmcid'] for row in rows]


narrowed = search_in_community(question, anchor_community)   # pool 은 기본값 20편
print('좁히기 전(상위 5):', [hit['pmcid'] for hit in hits])
print('좁힌 뒤       :', narrowed)

In [ ]:
# 편수를 외울 값이 아니다. 좁혀졌는지만 본다
print('상위 20편 중 남은 것:', len(narrowed), '편')

상위 20편이 예닐곱 편으로 줄었고, **2-3 에서 비율을 0.7 로 올렸을 때 1위로 올라왔던 타목시펜 논문이 여기서는 아예 빠졌습니다.** 그 논문이 다루는 개체가 이 묶음에 하나도 없기 때문입니다. 리랭킹은 순위를 흔들었고, 커뮤니티 필터는 **범위를 잘랐습니다.** 두 도구가 하는 일이 다릅니다.

> **남는 편수는 여러분 화면에서 조금 다를 수 있습니다**(실측 7편). 묶음 경계는 데이터를 다시 적재하면 달라질 수 있습니다(3-1). 편수는 흔들려도 **타목시펜 논문이 빠진다는 것은 그대로입니다.** 흔들리는 값과 흔들리지 않는 값을 갈라 읽어야 합니다.

> 좁히면 놓치는 것도 생깁니다. 정말 관련 있는 논문이 다른 묶음에 있으면 함께 빠집니다. **무엇을 물었는지에 따라 골라 써야 합니다.**

### 🖐️ 함께 따라하기: 다른 약을 기준으로 좁히기

이번에는 **`Omeprazole`**(위산 분비를 줄이는 약)을 기준으로 좁혀 봅니다.

1. `Omeprazole` 노드의 `community` 를 읽어 `follow_community` 에 담으세요.
2. 그 묶음에 개체가 몇 개 있는지 출력하세요.
3. 질문 '약물 상호작용은 어디에서 일어나나요?' 로 `search_in_community` 를 돌려, 2-3 따라하기의 상위 5편과 좁힌 뒤 목록을 나란히 출력하세요.

**확인 기준**: 상위 20편 중 **여덟 편 안팎**이 남습니다. 클로피도그렐 때보다 덜 좁혀지는데, 이 약이 속한 묶음이 더 크기 때문입니다. **편수를 맞히는 것이 목표가 아닙니다.** 좁혀졌는지, 그리고 남은 것이 그 갈래의 이야기인지를 보세요.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈 (3-1·3-2)

**1.** 커뮤니티 **번호(communityId)** 를 자가채점에서 정확한 값으로 비교하면 안 되는 이유는?

<details><summary>정답 보기</summary>

커뮤니티 **번호 자체에는 뜻이 없고**, 데이터를 다시 적재하면 같은 묶음이 다른 번호를 받습니다. 그래서 번호가 아니라 **"두 개체가 같은 묶음인가"** 같은 안정된 관계로 판단합니다.

</details>

**2.** 묶음이 백수십 개 나왔는데 "주제가 백수십 가지"라고 읽으면 안 되는 이유는?

<details><summary>정답 보기</summary>

그중 **100개 넘는 묶음이 혼자짜리**이기 때문입니다. 이 투영에 관계가 하나도 없는 노드가 각자 자기 묶음이 된 것뿐입니다. 묶음 수를 읽을 때는 **크기 분포를 함께** 봐야 합니다.

</details>

---
# 4. 찾은 근거로 답 만들기

> **기본 GraphRAG 로 돌아옵니다.** 최적화를 빼고 1절의 검색만으로 답을 한 번 만든 뒤, 2·3절에서 만든 최적화를 얹어 답이 어떻게 달라지는지 견줍니다.

앞의 두 절이 고른 논문을 **근거로 바꿔** 모델에게 넘기고, 그 답을 어디까지 믿을지 가릅니다.

- **4-1** 논문 발췌에 그 논문이 가리키는 개체를 붙여 근거를 만듭니다.
- **4-2** 그 근거를 붙여 모델이 답하게 합니다.
- **4-3** 답의 인용을 대조하고, 그 답이 말할 수 있는 것과 없는 것을 가릅니다.

## 4-1. 근거 만들기: 발췌 + 언급한 개체

> **먼저 2·3절을 쓰지 않고 답해 봅니다.** 벡터 검색 상위 세 편만 근거로 삼는 것이 가장 단순한 RAG 입니다. 그 답을 받아 두고, 4-2 뒤에서 **리랭킹으로 고른 근거**와 견줍니다. 무엇이 좋아지는지는 견줘 봐야 알 수 있으니까요.

### 오늘 새로운 것
검색해서 찾은 문서를 근거로 붙여 모델이 답하게 하는 **RAG** 는 앞 교과목에서 배웠습니다. **오늘 새로운 것은 그 근거를 그래프에서 뽑는다는 것입니다.** 그래서 앞의 두 절이 그대로 무기가 됩니다.

1. 질문으로 **논문을 검색**하고, 그래프 점수로 **순서를 다듬는다**.
2. 찾은 논문의 발췌에 **그 논문이 그래프에서 가리키는 개체**를 함께 붙여 근거로 만든다.
3. 모델이 **그 근거 안에서** 답한다.

> 3-3 의 커뮤니티 필터는 4절에서 쓰지 않습니다. 좁히기는 **범위를 자르는** 도구라 질문이 그 갈래 안에 있을 때만 맞는데, 4절 질문은 갈래를 가리지 않습니다. 여기서는 순위만 다듬는 2절 리랭킹을 씁니다.

<img src="images/그래프RAG_파이프라인.png" alt="질문에서 답까지의 흐름과 그래프가 끼어드는 세 자리" width="900">

In [ ]:
def build_context(pmcids):
    """논문 발췌와 그 논문이 언급한 개체를 이어 근거 문단으로 만든다."""
    # ORDER BY 없이 collect 하면 이름 순서가 보장되지 않아, 같은 질문인데 프롬프트가 달라질 수 있습니다
    rows = run_cypher('''MATCH (d:Document) WHERE d.pmcid IN $ids
                         OPTIONAL MATCH (d)-[:MENTIONS]->(e)
                         WITH d, e ORDER BY e.name
                         RETURN d.pmcid AS pmcid, d.title AS title, d.text AS text,
                                collect(DISTINCT e.name)[..8] AS entities''', ids=pmcids)
    by_id = {row['pmcid']: row for row in rows}
    parts = []
    for pmcid in pmcids:            # 넘긴 순서 그대로 이어 붙인다
        row = by_id[pmcid]
        # 본문을 통째로 넣습니다. 검색은 본문 전체로 했으니 근거도 전체라야 앞뒤가 맞습니다
        parts.append(f"[{row['pmcid']}] {row['title']}\n"
                     f"  본문: {row['text']}\n"
                     f"  이 논문이 언급한 개체: {', '.join(row['entities'])}")
    return '\n\n'.join(parts)


qa_question = '약물 대사 유전자 검사를 처방 전에 하면 무엇이 달라지나요?'
# 먼저 2·3절을 쓰지 않고, 1절의 벡터 검색 상위 3편만 근거로 삼아 봅니다
plain_docs = [hit['pmcid'] for hit in search_docs(qa_question, top_k=3)]
print('근거로 쓸 논문(벡터 검색만):', plain_docs)

In [ ]:
print(build_context(plain_docs)[:420], '...')

근거 문단에 **논문 발췌와 개체 목록이 함께** 들어갑니다. 개체 목록은 논문이 말한 것이 아니라 **우리가 그래프와 이어 둔 것**이라, 모델에게 "이 논문은 이 약·이 유전자 이야기다"를 알려 주는 셈입니다.

> 코드에서 눈여겨볼 데가 둘 있습니다. **`collect` 앞에 `ORDER BY` 를 붙인 것**과 **논문을 넘긴 순서대로 이어 붙인 것**입니다. 둘 다 없으면 이름과 문단 차례가 실행마다 달라져 **같은 질문인데 프롬프트가 매번 바뀝니다.** 답을 서로 비교하려면 프롬프트가 먼저 흔들리지 않아야 합니다.

> **본문을 자르지 않는 이유**: 우리는 제목+본문 **전체**를 임베딩해 검색했습니다. 근거로 앞부분만 주면 3,000자 지점 때문에 걸린 논문의 **그 근거를 모델이 못 봅니다.** 검색한 단위와 근거로 주는 단위를 같게 둡니다. 세 편을 통째로 넣어도 약 2,900 토큰이라 길이도 문제가 안 됩니다. 다만 오늘 논문이 짧아서 되는 것이고, 문서가 길면 **청크 단위로 임베딩해 걸린 청크를 근거로** 줍니다(16일차).

> `[..8]` 은 개체를 여덟 개까지만 싣습니다. 정렬이 이름 순이라 **잘리는 기준도 이름 순**입니다. 근거 첫 편은 개체 22개 중 8개만 실려 `Rifampicin`·`SLCO1B1` 같은 이름이 빠집니다. **무엇을 재서 자를지는 정해 두지 않았다**는 뜻입니다.

## 4-2. 근거를 붙여 답 만들기

근거가 준비됐으니 모델에게 넘깁니다. 프롬프트에 **규칙 두 가지**를 함께 적습니다. 발췌에 없는 내용을 지어내지 말 것, 문장마다 근거가 된 논문 번호를 달 것.

> 실무 프롬프트에는 규칙이 하나 더 붙습니다. **"발췌에 답할 내용이 없으면 없다고 답하세요."** 지어내지 말라는 금지만으로는 부족합니다. 모델에게 **빠져나갈 길**을 주지 않으면, 세 문장을 채우라는 지시를 지키려고 근거를 늘려 잡습니다. 오늘은 규칙 둘로 두고, 규칙을 더 넣었을 때 답이 어떻게 달라지는지는 직접 확인해 보세요.

프롬프트 틀·모델·파서를 파이프로 잇는 것까지 18일차 그대로입니다. 파서가 끝에 있으니 결과가 곧바로 문자열이라 `.text` 를 따로 부르지 않습니다.

In [ ]:
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [ ]:
# 근거를 붙여 모델에게 답을 시킵니다. 이것이 GraphRAG 입니다.
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# 18일차에서 쓴 그 프롬프트 틀입니다. {context}·{question} 이 나중에 채워질 빈칸입니다
qa_prompt = ChatPromptTemplate.from_messages([
    ('system', '주어진 논문 발췌 안에서만 답하는 조수다. 발췌에 없는 내용은 지어내지 않는다.'),
    ('human', '다음 논문 발췌를 근거로 질문에 한국어 세 문장 안으로 답하세요.\n'
              '각 문장 끝에 근거가 된 논문의 pmcid 를 대괄호로 다세요.\n\n'
              '[논문]\n{context}\n\n[질문] {question}'),
])


# 18일차처럼 파이프로 잇습니다: 빈칸 채우기 -> 모델 -> 글자만 남기기
qa_chain = qa_prompt | model | StrOutputParser()


def ask(question, pmcids):
    """찾은 논문을 근거로 붙여 모델의 답을 돌려준다."""
    return qa_chain.invoke({'context': build_context(pmcids), 'question': question})


plain_answer = ask(qa_question, plain_docs)   # 벡터 검색만으로 고른 근거로 답한다
print(plain_answer)

여기까지가 **그래프를 하나도 안 쓴 답**입니다. 근거를 벡터 유사도만으로 골랐습니다. 이제 2-3 의 리랭킹으로 근거를 다시 골라 같은 질문에 답해 봅니다.

In [ ]:
# 이번엔 2-3 의 리랭킹으로 근거를 고릅니다. 상위 5편을 받아 섞어 다시 세운 뒤 앞 3편만 씁니다
qa_docs = rerank(search_docs(qa_question, top_k=5), 0.3)[:3]
print('벡터 검색만 :', plain_docs)
print('리랭킹 뒤   :', qa_docs)

In [ ]:
answer = ask(qa_question, qa_docs)   # 답을 변수에 받아 둔다. 규칙을 지켰는지는 다음 셀이 이 답으로 대조한다
print()
print(answer)

**근거 세 편 가운데 하나가 바뀌었습니다.** 벡터 유사도만 보면 5위였던 타목시펜 논문이 그래프 점수(1.31, 다섯 편 중 가장 높음) 덕에 3위로 올라와 근거에 들어왔고, 대신 원래 3위였던 논문이 밀렸습니다. 근거가 바뀌면 답도 바뀝니다. 두 답을 나란히 읽어 보세요.

> **그럼 리랭킹이 늘 더 나은가요?** 아닙니다. 이 질문은 "유전자 검사를 처방 전에 하면 무엇이 달라지나" 인데, 타목시펜 논문이 근거로 들어온 것이 답에 보탬이 됐는지는 **읽어 봐야** 압니다. 2-3 에서 비율을 0.7 로 올렸을 때 그 논문이 1위까지 올라온 것을 봤죠. 그래프 점수를 너무 믿으면 질문에서 멀어집니다. **섞는 비율은 답을 읽어 가며 정하는 값**입니다.

방금 한 일이 **GraphRAG** 입니다. 검색이 문서에서 끝나지 않고 **그래프로 이어져** 순위와 근거를 함께 만든 것입니다. GraphRAG 라는 말은 넓게 쓰입니다. 그래프를 커뮤니티로 묶어 미리 요약해 두고 그 요약으로 답하는 방식도 GraphRAG 라 부릅니다. 오늘 것은 그 갈래가 아니라 **문서를 벡터로 찾고 그래프로 순위와 범위를 다듬는** 갈래입니다. 갈래를 나눠 견주는 일은 **42일차**에서 합니다.

## 4-3. 그 답을 어디까지 믿을 것인가

답이 나왔습니다. 그런데 **이 답을 어디까지 믿어야 할까요?** 두 가지를 확인합니다.

**첫째, 인용이 진짜인가.** 모델이 근거로 받지 않은 논문 번호를 붙일 수도 있습니다. 대조해 봅니다.

In [ ]:
# 답에 붙은 pmcid 가 실제로 근거로 넘긴 논문인지 대조합니다.
import re

# 앞 셀이 받아 둔 answer 를 그대로 대조한다. 모델을 다시 부르면 문장이 조금 달라질 수 있다
cited = set(re.findall(r'PMC\d+', answer))
print('답이 인용한 논문:', sorted(cited))
print('근거로 넘긴 논문:', sorted(qa_docs))
print('넘기지 않은 것을 인용했나:', sorted(cited - set(qa_docs)))

빈 목록이면 통과입니다. 비어 있지 않다면 그 답은 **쓰지 않습니다.** 근거로 준 적 없는 번호를 붙였다는 것은 모델이 자기가 아는 것을 섞었다는 뜻이라, 그 문장 하나만 지우는 것으로는 부족합니다. 실무에서는 이때 답을 버리고 다시 묻거나, 근거를 더 붙여 다시 받습니다. 그래서 이 대조는 **보고 넘기는 검사가 아니라 통과하지 못하면 답을 내보내지 않는 관문**입니다.

**둘째, 그 문장이 무엇을 말하는가.** 이쪽이 훨씬 중요합니다.

근거로 넘긴 것은 **논문이 보고한 내용**이지 **처방 지침이 아닙니다.** 인용 대조는 번호의 집합만 보므로 **"기여할 수 있다"가 "기여한다"로 넘어가는 것**을 잡지 못합니다. 위에서 받은 답을 그 눈으로 다시 읽어 보세요.

| 근거의 종류 | 무슨 뜻인가 | 답에 쓸 때 |
|---|---|---|
| 논문 발췌 | 그 연구가 **관찰하고 보고한 것** | "이 논문은 ...라고 보고했다" |
| 지식그래프 관계 | 2016년에 **문헌에서 정리된 것** | "문헌에는 ...로 정리돼 있다" |
| 둘 다 아닌 것 | 모델이 **채워 넣은 것** | 쓰지 않는다 |

> **36일차부터** 우리는 논문에서 사실을 뽑아 그래프에 넣습니다. 그때 관계마다 **이 사실이 어디서 왔는지**를 함께 적습니다. 오늘 본 이 구분이 그 설계의 이유입니다.

<img src="images/답을_내보내기_전_두관문.png" alt="인용 대조와 문장의 무게, 답을 내보내기 전에 거치는 두 관문" width="900">

### 🖐️ 함께 따라하기: 다른 질문에 답하게 하기

같은 파이프라인에 다른 질문을 넣습니다.

1. 질문 '여러 약을 함께 먹을 때 어떤 문제가 보고됐나요?' 로 `search_docs` 상위 5편을 받아 `rerank` 를 **`w=0.3`** 으로 걸고 **앞 3편**을 고르세요(데모와 같은 방식입니다).
2. `ask` 로 답을 받아 출력하세요.

**확인 기준**: 답이 **비어 있지 않고**, 근거 세 편 가운데 **최소 한 편**을 인용해야 합니다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈 (4-1·4-2·4-3)

**1.** 근거를 함께 넘기는데도 답을 그대로 믿으면 안 되는 이유는?

<details><summary>정답 보기</summary>

모델이 근거에 **없는 문장을 채워 넣을 수 있고**, 근거로 받지 않은 출처를 붙일 수도 있습니다. 그리고 근거가 정확하더라도 그것은 "논문이 그렇게 보고했다"까지이지 "그러니 이렇게 하면 된다"가 아닙니다.

</details>

**2.** 근거 문단에 논문 발췌뿐 아니라 **그 논문이 언급한 개체 목록**까지 넣은 이유는?

<details><summary>정답 보기</summary>

그 목록은 논문 본문이 아니라 **그래프와 이어 둔 결과**입니다. 본문에 안 나오는 개체까지 모델에게 알려 주고, 답이 어느 약·어느 유전자 이야기인지 붙들어 둘 수 있습니다.

</details>

---
## 🚀 응용 클론코딩: 근거를 대는 그래프 검색 도우미

오늘 만든 부품을 **한 함수로** 엮습니다. 질문을 받아 **답과 출처를 함께** 돌려주는 **`graph_answer(question, w=0.3)`** 를 완성하세요. 다섯 걸음이 어디서 이어지는지 짚어 보세요. 코드는 그보다 짧습니다.

1. `search_docs(question, top_k=5)` 로 후보를 받는다 (1-1)
2. `rerank(hits, w)` 로 순서를 다듬어 **앞 3편**만 남긴다 (2-3)
3. `ask` 가 안에서 `build_context` 를 불러 발췌와 언급한 개체를 근거로 만든다 (4-1). 그래서 우리가 따로 부를 것은 없고, **넘기는 것은 근거 문단이 아니라 `pmcid` 목록**이다.
4. `ask(question, docs)` 로 답을 받는다 (4-2)
5. 답이 인용한 `PMC` 번호가 근거 안에 있는지 대조해 함께 돌려준다 (4-3)

돌려주는 값은 `{'docs': 근거 목록, 'answer': 답, 'outside': 근거 밖 인용}` 딕셔너리로 하세요.

> **3-3 의 좁히기는 일부러 넣지 않습니다.** 넣으면 같은 질문에 근거가 매번 달라져, 이 함수가 돌려주는 답을 서로 견줄 수 없습니다(4절 도입에서 본 이유). 좁히기는 사람이 결과를 훑을 때 씁니다.

**확인 기준**: `graph_answer(qa_question)` 이 **4-2 에서 리랭킹으로 고른 근거 세 편(`qa_docs`)과 같은 목록**을 돌려주고, `outside` 가 **빈 목록**입니다. 4-2 와 같은 질문·같은 근거라 프롬프트는 같지만, 모델을 다시 부르므로 문장은 조금 달라질 수 있습니다. `outside` 가 비어 있지 않으면 그 답은 내보내지 않는다는 뜻으로 읽으세요. 함수가 답과 함께 이 목록을 돌려주는 이유가 그것입니다.

In [ ]:
# 여기에 코드를 작성하세요

---
## 이번 강의 정리

| 절 | 문법 | 핵심 |
|---|---|---|
| 1-1 | `VectorRetriever(...).search(query_text=..., top_k=...)` · `result_formatter` | 의미 검색을 부품으로 조립. 결과 모양은 우리가 정한다 |
| 1-2 | `doc_vec` + `doc_ft` | 한 인덱스만 쓰면 놓치는 것이 있다 |
| 2-1 | `gds.graph.project` · `gds.pageRank.write` | 개체에 중심성 점수를 새긴다 |
| 2-2 | `avg(e.pagerank)` | 개체 점수를 문서 점수로 옮긴다(투영 밖 개체는 분모에서 빠진다) |
| 2-3 | `minmax` + 가중합 | 눈금을 맞춰 섞는다. 비율은 재 봐야 안다 |
| 3-1 | `gds.leiden.write` | 개체를 묶는다. 묶음 수는 크기 분포와 함께 읽는다 |
| 3-2 | `MATCH (n) WHERE n.community = $c` | 묶음을 열어 무엇의 모임인지 확인한다 |
| 3-3 | 같은 묶음 개체를 언급한 논문만 | 순서가 아니라 **범위**를 자른다 |
| 4-1 | `build_context` | 발췌 + 그 논문이 가리키는 개체 |
| 4-2 | `ask` | 근거 안에서만 답하라는 규칙을 프롬프트에 |
| 4-3 | `re.findall(r'PMC\d+', answer)` | 인용이 근거 안에 있는지 대조 |

- 리랭킹과 필터는 **하는 일이 다릅니다.** 하나는 순서를 흔들고 하나는 후보를 잘라 냅니다.
- 섞는 비율에는 **적당한 값**이 있고, 그것은 재 봐야 압니다.
- 근거가 붙었다고 답이 참인 것은 아닙니다. **"논문이 보고했다"와 "효능이 입증됐다"는 다릅니다.**

## ⏭️ 예고: 다음 단원(36일차)

**36일차부터는** 문서에서 **개체와 관계를 뽑아내** 그래프를 스스로 만드는 쪽으로 넘어갑니다. 오늘은 이미 만들어진 그래프를 검색했다면, 다음에는 **그래프를 만드는 쪽**입니다. 오늘 이름 매칭이 놓친 것들을 모델이 어떻게 잡아내는지 보게 됩니다.

### 그리고 42일차: Text2Cypher

오늘은 우리가 Cypher 를 **직접** 짰습니다. 한 걸음 더 나아가면 사용자의 자연어 질문을 **모델이 Cypher 로 번역**하게 할 수 있습니다. 이것을 **Text2Cypher** 라고 합니다.

```
사용자: "클로피도그렐이 붙는 유전자가 뭐야?"
   (모델이 번역)
Cypher: MATCH (c:Compound {name:'Clopidogrel'})-[:BINDS]->(g:Gene) RETURN g.name
```

모델이 쓴 쿼리를 **그대로 실행하면 안 되기 때문에** 아래 그림처럼 관문을 둡니다. 구현과 안전장치는 **42일차**에서 직접 만들어 봅니다.

<img src="images/Text2Cypher_안전장치.png" alt="모델이 만든 Cypher 를 실행하기 전에 거치는 세 겹의 안전장치" width="900">

수고하셨습니다!